<a href="https://colab.research.google.com/github/EiMonSan-Ellie/LLM_RAG-Project/blob/main/Innovative_Research_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Business Context

At Innovative Research Labs, a research organization focused on advancing LLM-based technologies, staying abreast of the latest technological developments is crucial for fostering innovation and progressing projects. By streamlining access to essential information, the organization can maintain its competitive edge in the rapidly evolving landscape of LLM technologies, driving future innovations and advancements within the industry.

However, researchers often face the challenge of sifting through a vast number of articles and publications to extract valuable insights for their work. The sheer volume of information can make it difficult to quickly locate specific details or fully comprehend complex concepts, such as the Transformer architecture, which is fundamental to their research efforts.

This case study demonstrates how a document question-answering system can effectively extract relevant insights from Jay Alammar's blog article, "The Illustrated Transformer." This use case highlights the significant advantages of leveraging Generative AI to enhance research capabilities and streamline knowledge retrieval processes.

#**LangChain Document Question Answering System**

In [ ]:
!pip install transformers faiss-cpu sentence-transformers

In [ ]:
!pip freeze > requirement.txt

## **The LangChain Pipeline**

The pipeline for converting raw unstructured data into a QA chain looks like the following:

- **Loading:** First we need to load our data. Unstructured data can be loaded from many sources. The LangChain integration hub contains the full set of loaders. Each loader returns data as a LangChain Document.
- **Splitting:** Text Splitters break Documents into splits of specified size
- **Storage:** Storage (ex: often a Vector Store) will house and often embed the splits
- **Retrieval:** The app retrieves splits from storage (ex: often with similar embeddings to the input question)
- **Generation:** An LLM produces an answer using a prompt that includes the question and the retrieved data
- **Conversation (Extension):** Hold a multi-turn conversation by adding Memory to your QA chain.

## **Step 1: Loading**

In [ ]:
from langchain.document_loaders import WebBaseLoader

loader = WebBaseLoader("http://jalammar.github.io/illustrated-transformer/")
data = loader.load()

## **Step 2: Splitting**









In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 0)
all_splits = text_splitter.split_documents(data)

## **Step 3: Storing**

In [ ]:
# Import FAISS from Langchain Vectorstore
from langchain.vectorstores import FAISS

In [ ]:
from langchain.llms import HuggingFacePipeline
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
hf = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

In [ ]:
 # Creating a vector store
vectorstore = FAISS.from_documents(documents=all_splits, embedding=hf) ## hf are the hugging face embeddings

## **Step 4: Retrieval**



In [ ]:
question = "What are transformers?"
docs = vectorstore.similarity_search(question)
docs

## **Step 5: Generation**

In [ ]:
question = "What are transformers?"

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA

# Load model and tokenizer
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Define the pipeline
hf_pipeline = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=512)

# Wrap the pipeline in LangChain's LLM class
llm = HuggingFacePipeline(pipeline=hf_pipeline)

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(llm,retriever=vectorstore.as_retriever())
qa_chain({"query": question})

In [ ]:
question = " What is attention mechanism?"
qa_chain({"query": question})

## **Step 6: Chat**


## **Conversation Summary Memory**

There are different types of memory. Each has their own parameters, their own return types, and is useful in different scenarios

We will be using **Conversation Summary Memory**. This type of memory creates a summary of the conversation over time, which can be useful for condensing information from the conversation over time.

**Conversation Summary Memory** summarizes the conversation as it happens and stores the current summary in memory. This memory can then be used to inject the summary of the conversation so far into a prompt/chain. This memory is most useful for longer conversations, where keeping the past message history in the prompt verbatim would take up too many tokens.

In [ ]:
from langchain.memory import ConversationSummaryMemory

In [ ]:
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    return_messages=True
)

## **Conversational Retrieval Chain**

This is a type of chain for having a conversation based on retrieved documents. This chain takes in chat history (a list of messages) and new questions, and then returns an answer to that question. The algorithm for this chain consists of three parts:

1. **Use the chat history and the new question to create a “standalone question”.** This is done so that this question can be passed into the retrieval step to fetch relevant documents. If only the new question was passed in, then the relevant context may be lacking. If the whole conversation was passed into retrieval, there may be unnecessary information there that would distract from retrieval.

2. **This new standalone question is passed to the retriever**, and relevant documents are returned.

3. **The retrieved documents are passed to an LLM** along with either the new question (default behavior) or the original question and chat history to generate a final response.



In [ ]:
from langchain.chains import ConversationalRetrievalChain

retriever = vectorstore.as_retriever()
chat = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=retriever,
    memory=memory,
    verbose=True
)

## **Demonstration**

In [ ]:
chat("Explain self-attention")

In [ ]:
chat("What is a gentler approach to transformers?")

In [ ]:
chat("Where were transformers proposed?")

In [ ]:
chat("What are the different layers in a typical Transformer model?")

In [ ]:
chat("If the vocabulary is 10,000 words, what would the width of the logits vector?")

In [ ]:
chat("Explain the training process of a Transformer network in detail")

#**Conclusion**

The implementation of a document question-answering system at Innovative Research Labs has demonstrated significant potential for enhancing research efficiency and productivity. By utilizing Generative AI to extract relevant insights from complex articles such as Jay Alammar's "The Illustrated Transformer," researchers can overcome the challenges posed by information overload. This technology not only accelerates the knowledge retrieval process but also allows researchers to focus on deeper analysis and innovation.